# Roboat A/B training on Colab (yolov8n, 2-class)

Floating-litter + water-hyacinth YOLO detector for a USV on Bengaluru lakes.
Runs the two controlled runs from `kaggle/RUN_PLAN.md` on the verified 2-class
dataset (`data/processed/merged2`, classes `0=litter`, `1=hyacinth`).

**Locked settings (do not tune):** `seed=42`, `epochs=100`, `patience=20`,
`imgsz=960` (D11 box-size audit), `deterministic=True`.

| Run | Init | Notes |
|-----|------|-------|
| A | COCO (`yolov8n.pt`) | baseline |
| B | Zenodo `Model_tiles_weights_converted.pt` (nc=3) | Ultralytics auto-transfers **349/355** state-dict keys into the nc=2 model, skipping exactly the 6 `model.22.*` Detect cls-conv keys (verified, see RUN_PLAN.md) |

## Colab caveats (from `kaggle/RUN_PLAN.md` §Colab caveat)

- **Wall-clock budget:** one run trains the RFS-expanded 10,070-image list at
  imgsz 960 on a T4 ≈ 15–20 min/epoch → 100 epochs ≈ **25–35 h per run**. That
  does not fit one free Colab session (~12 h) or one Kaggle GPU session
  (~9–12 h): plan **~2–3 free sessions per run with resume**, or run on Kaggle
  (~30 GPU-h/week).
- **If a session dies, resume — don't restart.** Runs write to Drive
  (`project=/content/drive/MyDrive/roboat_runs`), so `last.pt` survives the
  session. In a fresh session, re-run cells 1–2c, then the §3/§4 training
  cells auto-detect `last.pt` and call `train(resume=True)`; there is also a
  dedicated "Resume an interrupted run" cell after §4. On Kaggle the
  equivalent is re-attaching the previous session's output as an input and
  calling `train(resume=True)` on its `last.pt` (or
  `python train_baseline.py --resume`).
- **P2 note:** the val split is *same-site* and will flatter the model. The
  Bengaluru Capture Set (T9) is the real test — score it BEFORE any fine-tune on it.
- License: Ultralytics-pretrained weights (incl. `yolov8n.pt` and anything derived
  from the Zenodo checkpoints) are **AGPL-3.0** — see RUN_PLAN.md §License note.

Run the cells top-to-bottom. Every cell is small and re-runnable.


In [ ]:
# 1. Setup
%pip install -q ultralytics

import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > Hardware accelerator > T4 GPU.")

## 2. Data

Upload the Kaggle-style dataset zip produced by `scripts/build_kaggle_bundle.py`
(`roboat-processed.zip`, top level: `merged2/ lists/ yamls/ ood_aquatrash/
zenodo_12800597/converted/` + `BUNDLE_README.txt` — layout per
`kaggle/README.md` §1; the `ood_aquatrash` OOD tree (369 images) is bundled
for the D6 OOD benchmark).

**Option A (recommended):** put `roboat-processed.zip` in your Drive root
`MyDrive/`, then run the next cell.

**Option B:** skip Drive and use the upload cell after it (~3.7k FML + 1.2k
TUD-GV + 1.2k Hagenbeek JPEGs; the upload can take a while).

In [ ]:
# 2a. Data from Google Drive (Option A)
USE_DRIVE = True  # set False if you use the upload cell instead

from pathlib import Path
DATA_ROOT = Path("/content/roboat-processed")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = Path("/content/drive/MyDrive/roboat-processed.zip")
    assert zip_path.exists(), f"{zip_path} not found - upload it to MyDrive first"
    !unzip -q -o {zip_path} -d /content/roboat-processed/

for d in ("merged2", "lists", "yamls", "ood_aquatrash", "zenodo_12800597/converted"):
    p = DATA_ROOT / d
    assert p.exists(), f"missing {p} - wrong zip?"
print("dataset OK at", DATA_ROOT)
print("train list lines:", len((DATA_ROOT / "lists/train.txt").read_text().splitlines()))

In [ ]:
# 2b. Data by direct upload (Option B) - only if USE_DRIVE was False
USE_UPLOAD = not USE_DRIVE

from pathlib import Path
DATA_ROOT = Path("/content/roboat-processed")

if USE_UPLOAD:
    from google.colab import files
    up = files.upload()  # pick roboat-processed.zip
    !unzip -q -o /content/roboat-processed.zip -d /content/roboat-processed/

for d in ("merged2", "lists", "yamls", "ood_aquatrash", "zenodo_12800597/converted"):
    p = DATA_ROOT / d
    assert p.exists(), f"missing {p} - wrong zip?"
print("dataset OK at", DATA_ROOT)

In [ ]:
# 2c. Rebase yaml/list paths onto the Colab mount point.
# Port of rebase() from kaggle/train_baseline.py:27-58: build_dataset.py writes
# ROOT-relative paths ("data/processed/merged2/..."), so map whatever follows
# "data/processed/" onto DATA_ROOT. Yaml path/train/val/test get the same
# treatment (the shipped yamls carry absolute C:/Users/... paths, which the
# marker also catches).
import yaml
from pathlib import Path

def rebase(data_root: Path, workdir: Path):
    lists = workdir / "lists"
    yamls = workdir / "yamls"
    lists.mkdir(parents=True, exist_ok=True)
    yamls.mkdir(parents=True, exist_ok=True)
    marker = "data/processed/"
    for f in (data_root / "lists").glob("*.txt"):
        out = []
        for line in f.read_text().splitlines():
            if marker in line:
                out.append(f"{data_root.as_posix()}/{line.split(marker)[-1]}")
            elif line:
                out.append(f"{data_root.as_posix()}/{line}")
        (lists / f.name).write_text("\n".join(out) + "\n")
    for f in (data_root / "yamls").glob("*.yaml"):
        y = yaml.safe_load(f.read_text())
        for key in ("train", "val", "test"):
            v = y.get(key)
            if v:
                sv = str(v).replace("\\", "/")
                y[key] = (f"{data_root.as_posix()}/{sv.split(marker)[-1]}"
                          if marker in sv else
                          (f"{data_root.as_posix()}/{sv}" if sv.startswith("data/") else sv))
        y["path"] = data_root.as_posix()
        (yamls / f.name).write_text(yaml.safe_dump(y))
    return yamls

REBASED = rebase(DATA_ROOT, Path("/content/rebased"))
TRAIN_YAML = REBASED / "combined_c2.yaml"  # 2-class: 0=litter, 1=hyacinth
print(TRAIN_YAML.read_text())

## 3. Run A — yolov8n, COCO init

`yolov8n.pt` stock COCO weights, trained on `combined_c2` with the locked
settings. A run takes ≈25–35 h at imgsz 960 on a T4 — far past one free
session — so the run writes straight to Drive
(`project=/content/drive/MyDrive/roboat_runs`; the §2a data cell mounts Drive
for Option A, and the cell below mounts it itself for Option B) and
**auto-resumes**: if the run's `last.pt` exists on Drive
(`/content/drive/MyDrive/roboat_runs/runA_coco_init/weights/last.pt`), it
loads it with `YOLO(last.pt)` and calls `.train(resume=True)` instead of
starting fresh.

**Resume note (applies to Run A and Run B):** `deterministic=True` +
`resume=True` restores the RNG state, so a resumed run continues the same
augmentation/schedule sequence as an uninterrupted one. Because the run dir
lives on Drive, `last.pt` survives session death with no copy-back step — in
a fresh session, re-run cells 1–2c, then this cell (or the dedicated resume
cell after §4). On Kaggle, `/kaggle/working` persists between Save-Version
runs: re-attach the previous session's output as an input and call
`train(resume=True)` on its `last.pt`, or run
`python train_baseline.py --resume`.


In [ ]:
# 3. Run A (COCO init) - writes to Drive; auto-resumes from last.pt if interrupted
from pathlib import Path
from ultralytics import YOLO

RUNS = "/content/drive/MyDrive/roboat_runs"
if not Path(RUNS).parent.exists():  # Option B never mounts Drive in the data cell
    from google.colab import drive
    drive.mount("/content/drive")
last_a = Path(RUNS) / "runA_coco_init" / "weights" / "last.pt"

if last_a.exists():
    print("resuming Run A from", last_a)
    model_a = YOLO(str(last_a))  # YOLO loader only - never raw torch.load
    model_a.train(resume=True)   # deterministic=True restores the RNG state
else:
    model_a = YOLO("yolov8n.pt")  # COCO init
    model_a.train(data=str(TRAIN_YAML), epochs=100, patience=20, imgsz=960,
                  seed=42, deterministic=True,
                  project=RUNS, name="runA_coco_init")
best_A = model_a.trainer.best
print("Run A best:", best_A)


## 4. Run B — yolov8n, init from converted Model_tiles checkpoint

Starts from
`zenodo_12800597/converted/Model_tiles_weights_converted.pt` (nc=3:
`ff_litter/hyacinth/ent_litter`, converted with the `YOLO()` loader by
`scripts/convert_zenodo_weights.py` — never raw `torch.load`). Ultralytics
transfers the compatible layers automatically: **349/355** state-dict keys
into the nc=2 model, skipping exactly the 6 `model.22.*` Detect cls-conv
keys. No manual surgery needed. The converted checkpoint is a yolov8n
architecture, so Run B is the yolov8n leg only.

Same auto-resume behavior as Run A (runs write to Drive): if
`/content/drive/MyDrive/roboat_runs/runB_zenodo_init/weights/last.pt`
exists, the cell below resumes from it instead of starting fresh (see the
resume note in §3).


In [ ]:
# 4. Run B (Zenodo init) - writes to Drive; auto-resumes from last.pt if interrupted
from pathlib import Path
from ultralytics import YOLO

CKPT = DATA_ROOT / "zenodo_12800597/converted/Model_tiles_weights_converted.pt"
assert CKPT.exists(), f"{CKPT} missing - is it in the bundle zip?"

last_b = Path(RUNS) / "runB_zenodo_init" / "weights" / "last.pt"

if last_b.exists():
    print("resuming Run B from", last_b)
    model_b = YOLO(str(last_b))  # YOLO loader only - never raw torch.load
    model_b.train(resume=True)   # deterministic=True restores the RNG state
else:
    model_b = YOLO(str(CKPT))  # YOLO loader only - never raw torch.load
    model_b.train(data=str(TRAIN_YAML), epochs=100, patience=20, imgsz=960,
                  seed=42, deterministic=True,
                  project=RUNS, name="runB_zenodo_init")
best_B = model_b.trainer.best
print("Run B best:", best_B)


### Resume an interrupted run

If a session died mid-run: in the fresh session, re-run cells 1–2c, then run
this cell — it reloads the run's `last.pt` **from Drive** and continues the
schedule. The same works for **Run B**: point `YOLO()` at
`/content/drive/MyDrive/roboat_runs/runB_zenodo_init/weights/last.pt`. On
Kaggle the equivalent is re-attaching the previous session's output as an
input and calling `model.train(resume=True)` on its `last.pt`
(`kaggle/train_baseline.py --resume` does exactly this).

In [ ]:
# 4b. Resume Run A from Drive (for Run B, swap in runB_zenodo_init/weights/last.pt)
from pathlib import Path
from ultralytics import YOLO

RUNS = "/content/drive/MyDrive/roboat_runs"
if not Path(RUNS).parent.exists():
    from google.colab import drive
    drive.mount("/content/drive")

model = YOLO(f"{RUNS}/runA_coco_init/weights/last.pt")
model.train(resume=True)  # deterministic=True restores the RNG state

## 5. Eval — P1 gate: per-class AND per-source test mAP@0.5

The P1 gate (RUN_PLAN.md) requires **test-split mAP50 reported per class AND
per source** (`saigon_tiles_c2`, `hagenbeek_tiles_c2`, `fml_c2`,
`tud_gv_c2`, `combined_c2`, `ood_aquatrash`), never one aggregate. The gate
reports on the **TEST split (1,160 images)**, unified with
`kaggle/train_baseline.py`, because the per-source VAL sets are too small for
stable per-class numbers (Hagenbeek hyacinth has just 64 val instances); val
remains the early-stopping split only. This cell validates each
run's `best.pt` on every per-source yaml and prints the two tables, then
saves everything to a results JSON (test-split numbers). A source
whose yaml is missing or fails to validate is skipped with a printed note —
one bad source never loses the whole report (mirrors
`kaggle/train_baseline.py:116-131`).

`ood_aquatrash` is the **D6 out-of-domain benchmark** (AquaTrash — never
trained on): expect the worst numbers here **by design**.

Caveat: per-source test sets are small for some sources — hagenbeek has 127
test tiles — so per-class numbers there are directional, not definitive.


In [ ]:
# 5. Per-source + per-class test eval (P1 gate) - split unified on "test"
import json
from pathlib import Path
from ultralytics import YOLO

PER_SOURCE = ["saigon_tiles_c2", "hagenbeek_tiles_c2", "fml_c2",
              "tud_gv_c2", "combined_c2", "ood_aquatrash"]

def map50_row(metrics):
    """Aggregate + per-class mAP@0.50 (port of kaggle/train_baseline.py:64-71)."""
    names = metrics.names
    idx = metrics.box.ap_class_index
    ap50 = metrics.box.ap50
    per_class = ({names[int(c)]: float(ap) for c, ap in zip(idx, ap50)}
                 if idx is not None and len(idx) else {})
    return {"all": float(metrics.box.map50), "per_class": per_class}

runs = {}
if Path(best_A).exists(): runs["runA_coco_init"] = best_A
if Path(best_B).exists(): runs["runB_zenodo_init"] = best_B

results = {}
for rname, best in runs.items():
    results[rname] = {}
    for ys in PER_SOURCE:
        yf = REBASED / f"{ys}.yaml"
        if not yf.exists():
            # a per-source yaml missing from the bundle - skip it, never lose
            # the whole report (mirrors kaggle/train_baseline.py:135-154)
            print(f"note: skipping per-source eval on {ys}: {yf} not found")
            continue
        try:
            m = YOLO(best)
            metrics = m.val(data=str(yf), imgsz=960, split="test")
        except Exception as exc:  # one bad source must not lose the report
            print(f"note: per-source eval on {ys} failed ({exc}); "
                  f"continuing with the remaining sources")
            continue
        results[rname][ys] = map50_row(metrics)

# aggregate per-source table (only sources that produced metrics for every run)
sources = [y for y in PER_SOURCE
           if results and all(y in row for row in results.values())]
print("===== test mAP@0.5 per source (aggregate) =====")
print(f"{'run':<20}" + "".join(f"{y:<22}" for y in sources))
for rname, row in results.items():
    print(f"{rname:<20}" + "".join(f"{row[y]['all']:<22.4f}" for y in sources))

# per-class per-source table (the P1 gate)
print("\n===== per-class test mAP@0.5 per source (P1 gate) =====")
for rname, row in results.items():
    for ys, vals in row.items():
        pc = "  ".join(f"{cls}={ap:.4f}" for cls, ap in vals["per_class"].items())
        print(f"{rname:<20}{ys:<22}{pc}")

json_path = Path(RUNS) / "p1_gate_results.json"
json_path.write_text(json.dumps(results, indent=2))
print("\nsaved:", json_path)

## 6. Save checkpoints to Drive

Runs already write to Drive (`project=/content/drive/MyDrive/roboat_runs`),
so `best.pt` and the resume checkpoint `last.pt` survive the session without
this cell. It additionally saves flattened copies (`<run>_best.pt`,
`<run>_last.pt`) and the P1 results JSON into `MyDrive/roboat_runs/` for
convenient download.


In [ ]:
# 6. Copy best.pt + last.pt + results JSON to Drive
import shutil
from pathlib import Path

dst = Path("/content/drive/MyDrive/roboat_runs")
if not dst.parent.exists():
    from google.colab import drive
    drive.mount("/content/drive")
dst.mkdir(parents=True, exist_ok=True)

for rname, best in runs.items():
    shutil.copy2(best, dst / f"{rname}_best.pt")
    print("saved:", dst / f"{rname}_best.pt")
for rname, last in (("runA_coco_init", last_a), ("runB_zenodo_init", last_b)):
    if Path(last).exists():
        shutil.copy2(last, dst / f"{rname}_last.pt")
        print("saved:", dst / f"{rname}_last.pt")
results_json = dst / "p1_gate_results.json"
if json_path != results_json:
    shutil.copy2(json_path, results_json)  # no-op once RUNS == dst (runs on Drive)
print("saved:", results_json)
